# Prototypical Networks with XAI for Few-Shot Learning

**Research-Grade Implementation**

This notebook implements:
- **Prototypical Networks** (Snell et al., 2017) for few-shot learning
- **Explainable AI (XAI)** techniques: Grad-CAM, Saliency Maps, Integrated Gradients
- **Comprehensive Evaluation**: Accuracy, F1-score, ECE, Brier Score, Attribution Sparsity
- **Statistical Significance Testing**: Multiple independent runs

---

## Mathematical Formulations

### 1. Prototype Computation

$$p_c = \frac{1}{|S_c|} \sum_{(x_i, y_i) \in S_c} f_\phi(x_i)$$

### 2. Distance Metric

$$d(z, p_c) = \|f_\phi(z) - p_c\|_2^2$$

### 3. Classification

$$P(y=c|z) = \frac{\exp(-d(z, p_c))}{\sum_{c'} \exp(-d(z, p_{c'}))}$$

### 4. Loss Function

$$\mathcal{L} = -\frac{1}{|Q|} \sum_{(z_j, y_j) \in Q} \log P(y=y_j|z_j)$$

---

## Dataset Structure

```
/kaggle/input/few-shot-data/
├── class_0/
│   ├── img_001.jpg
│   ├── img_002.jpg
│   └── ...
├── class_1/
│   ├── ...
└── ...
```

8 classes × 160 images = 1,280 total images

**Split:** 80% Train / 10% Val / 10% Test

## 1. Setup and Configuration

In [ ]:
# Install required packages
!pip install -q scikit-learn matplotlib seaborn tqdm opencv-python

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os
import sys
import random
import json
import math
import time
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Callable
from collections import defaultdict
from dataclasses import dataclass, asdict, field

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedShuffleSplit
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

# Configuration
@dataclass
class Config:
    data_root: str = '/kaggle/input/few-shot-data'
    output_dir: str = '/kaggle/working'
    n_classes: int = 8
    train_ratio: float = 0.8
    val_ratio: float = 0.1
    test_ratio: float = 0.1
    img_size: int = 128
    n_way: int = 8
    k_shot: int = 5
    q_query: int = 15  # Query for training
    q_query_eval: int = 5  # Query for val/test (smaller due to limited samples)
    n_epochs: int = 30
    lr: float = 1e-3
    weight_decay: float = 1e-4
    embedding_dim: int = 128
    episodes_per_epoch: int = 20
    val_episodes: int = 10
    test_episodes: int = 15
    seed: int = 42
    xai_samples: int = 10

CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create output directories
for d in ['splits', 'plots', 'xai', 'checkpoints', 'logs']:
    os.makedirs(os.path.join(CFG.output_dir, d), exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Device: {DEVICE}")
print(f"✓ Train: {CFG.k_shot}-shot, {CFG.q_query}-query")
print(f"✓ Val/Test: {CFG.k_shot}-shot, {CFG.q_query_eval}-query")

## 2. Dataset Handling with Stratified Splitting

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

def make_stratified_splits(config=CFG):
    """Create stratified train/val/test splits."""
    data = []
    root = Path(config.data_root)
    
    if not root.exists():
        raise FileNotFoundError(f"Data root not found: {config.data_root}")
    
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    
    for cls_name in classes:
        cls_path = root / cls_name
        images = [f for f in cls_path.glob('*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
        for img_path in images:
            data.append({
                'image_path': str(img_path),
                'label': class_to_idx[cls_name],
                'class_name': cls_name
            })
    
    df = pd.DataFrame(data)
    print(f"Total samples: {len(df)}")
    
    # Stratified split
    X, y = df['image_path'].values, df['label'].values
    
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(1-config.train_ratio), random_state=config.seed)
    train_idx, temp_idx = next(sss1.split(X, y))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_temp = df.iloc[temp_idx].reset_index(drop=True)
    
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=config.test_ratio/(config.val_ratio+config.test_ratio), 
                                   random_state=config.seed)
    X_temp, y_temp = df_temp['image_path'].values, df_temp['label'].values
    val_idx, test_idx = next(sss2.split(X_temp, y_temp))
    
    df_val = df_temp.iloc[val_idx].reset_index(drop=True)
    df_test = df_temp.iloc[test_idx].reset_index(drop=True)
    
    # Save splits
    df_train.to_csv(f"{config.output_dir}/splits/train.csv", index=False)
    df_val.to_csv(f"{config.output_dir}/splits/val.csv", index=False)
    df_test.to_csv(f"{config.output_dir}/splits/test.csv", index=False)
    
    print(f"\nSplit sizes: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")
    return df_train, df_val, df_test

# Create splits
df_train, df_val, df_test = make_stratified_splits()
print(f"\nClass distribution in training set:")
print(df_train['class_name'].value_counts().sort_index())

## 3. Data Loading and Episodic Sampling

In [ ]:
class FewShotDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        image = transforms.ToTensor()(image)
        if self.transform:
            image = self.transform(image)
        return image, int(row['label'])

class EpisodicSampler:
    """Sampler for N-way K-shot episodes."""
    def __init__(self, labels, n_way, k_shot, q_query, n_episodes, seed=42):
        self.labels = np.array(labels)
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.n_episodes = n_episodes
        self.rng = np.random.RandomState(seed)
        
        self.class_indices = {}
        for label in np.unique(self.labels):
            self.class_indices[label] = np.where(self.labels == label)[0]
    
    def __len__(self):
        return self.n_episodes
    
    def __iter__(self):
        for _ in range(self.n_episodes):
            selected_classes = self.rng.choice(list(self.class_indices.keys()), 
                                               size=self.n_way, replace=False)
            support_idx, query_idx = [], []
            for cls in selected_classes:
                n_samples = self.k_shot + self.q_query
                sampled = self.rng.choice(self.class_indices[cls], size=n_samples, replace=False)
                support_idx.extend(sampled[:self.k_shot])
                query_idx.extend(sampled[self.k_shot:])
            yield support_idx, query_idx

# Transforms
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.Normalize(mean=mean, std=std)
])

print("✓ Dataset and sampler ready")

## 4. Prototypical Network Model

In [ ]:
class ConvEncoder(nn.Module):
    """CNN encoder for embedding images."""
    def __init__(self, out_dim=128, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout/2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout/2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, out_dim)
        )
    
    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)  # L2 normalization
        return x

class PrototypicalNetwork(nn.Module):
    """Prototypical Network for few-shot learning."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
    
    def forward(self, support, support_labels, query):
        # Embed support and query
        z_support = self.encoder(support)   # [N*K, D]
        z_query = self.encoder(query)       # [N*Q, D]
        
        # Compute prototypes
        unique_labels = torch.unique(support_labels)
        prototypes = []
        for label in unique_labels:
            class_mask = (support_labels == label)
            prototype = z_support[class_mask].mean(dim=0)
            prototypes.append(prototype)
        prototypes = torch.stack(prototypes)  # [N, D]
        
        # Compute distances
        dists = self.euclidean_dist(z_query, prototypes)
        logits = -dists  # Closer = higher score
        return logits, prototypes, z_query
    
    @staticmethod
    def euclidean_dist(x, y):
        n = x.size(0)
        m = y.size(0)
        xx = (x**2).sum(dim=1, keepdim=True).expand(n, m)
        yy = (y**2).sum(dim=1, keepdim=True).expand(m, n).t()
        dist = xx + yy - 2.0 * x @ y.t()
        return torch.clamp(dist, min=0.0)

# Initialize model
encoder = ConvEncoder(out_dim=CFG.embedding_dim)
model = PrototypicalNetwork(encoder).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model initialized")
print(f"✓ Trainable parameters: {n_params:,}")

## 5. Training Loop

In [ ]:
# Training setup
train_dataset = FewShotDataset(df_train, transform=train_transform)
val_dataset = FewShotDataset(df_val, transform=eval_transform)

train_sampler = EpisodicSampler(df_train['label'].values, CFG.n_way, CFG.k_shot, 
                                 CFG.q_query, CFG.episodes_per_epoch, seed=CFG.seed)
# Use q_query_eval for validation (fewer samples needed)
val_sampler = EpisodicSampler(df_val['label'].values, CFG.n_way, CFG.k_shot,
                               CFG.q_query_eval, CFG.val_episodes, seed=CFG.seed)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0

print(f"\n{'='*60}")
print("TRAINING START")
print(f"{'='*60}")

for epoch in range(1, CFG.n_epochs + 1):
    # Training
    train_loss = AverageMeter()
    train_acc = AverageMeter()
    for support_idx, query_idx in train_sampler:
        loss, acc, _, _ = run_episode(model, optimizer, train_dataset, 
                                       support_idx, query_idx, training=True)
        train_loss.update(loss)
        train_acc.update(acc)
    
    scheduler.step()
    
    # Validation
    val_loss = AverageMeter()
    val_acc = AverageMeter()
    model.eval()
    with torch.no_grad():
        for support_idx, query_idx in val_sampler:
            loss, acc, _, _ = run_episode(model, optimizer, val_dataset,
                                           support_idx, query_idx, training=False)
            val_loss.update(loss)
            val_acc.update(acc)
    
    # Record
    history['train_loss'].append(train_loss.avg)
    history['train_acc'].append(train_acc.avg)
    history['val_loss'].append(val_loss.avg)
    history['val_acc'].append(val_acc.avg)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    print(f"Epoch {epoch:2d}/{CFG.n_epochs} | "
          f"Train Loss: {train_loss.avg:.4f} Acc: {train_acc.avg:.3f} | "
          f"Val Loss: {val_loss.avg:.4f} Acc: {val_acc.avg:.3f}")
    
    # Save best
    if val_acc.avg > best_val_acc:
        best_val_acc = val_acc.avg
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': best_val_acc
        }, f"{CFG.output_dir}/checkpoints/best_model.pth")

print(f"\n{'='*60}")
print(f"Training complete! Best Val Acc: {best_val_acc:.4f}")
print(f"{'='*60}")

## 6. Evaluation and Metrics

In [ ]:
# Load best model
checkpoint = torch.load(f"{CFG.output_dir}/checkpoints/best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluation function
def compute_ece(probs, labels, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels)
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            acc_bin = np.mean(accuracies[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += np.abs(conf_bin - acc_bin) * prop_in_bin
    return ece

def evaluate_model(model, df, split='test'):
    test_dataset = FewShotDataset(df, transform=eval_transform)
    # Use q_query_eval for test (fewer samples needed)
    test_sampler = EpisodicSampler(df['label'].values, CFG.n_way, CFG.k_shot, 
                                    CFG.q_query_eval, CFG.test_episodes, seed=CFG.seed)
    
    all_preds, all_labels, all_probs, all_loss = [], [], [], []
    
    model.eval()
    with torch.no_grad():
        for support_idx, query_idx in test_sampler:
            loss, acc, preds, labels = run_episode(model, None, test_dataset,
                                                    support_idx, query_idx, training=False)
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_loss.append(loss)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro'),
        'f1_micro': f1_score(all_labels, all_preds, average='micro'),
        'precision_macro': precision_score(all_labels, all_preds, average='macro'),
        'recall_macro': recall_score(all_labels, all_preds, average='macro'),
        'precision_per_class': precision_score(all_labels, all_preds, average=None).tolist(),
        'recall_per_class': recall_score(all_labels, all_preds, average=None).tolist(),
        'f1_per_class': f1_score(all_labels, all_preds, average=None).tolist(),
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist(),
    }
    
    return metrics

# Evaluate
metrics = evaluate_model(model, df_test, 'test')

# Save metrics
with open(f"{CFG.output_dir}/test_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n{'='*60}")
print("TEST SET RESULTS")
print(f"{'='*60}")
print(f"Accuracy:         {metrics['accuracy']:.4f}")
print(f"F1-Score (Macro): {metrics['f1_macro']:.4f}")
print(f"F1-Score (Micro): {metrics['f1_micro']:.4f}")
print(f"Precision (Macro): {metrics['precision_macro']:.4f}")
print(f"Recall (Macro):   {metrics['recall_macro']:.4f}")
print(f"{'='*60}")

## 7. Visualizations

In [ ]:
# Training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs = range(1, len(history['train_loss']) + 1)

axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Loss over Epochs')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train', linewidth=2)
axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy over Epochs')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

gap = [v - t for t, v in zip(history['train_acc'], history['val_acc'])]
axes[1, 1].bar(epochs, gap, color=['green' if g > 0 else 'red' for g in gap], alpha=0.7)
axes[1, 1].axhline(y=0, color='black', linestyle='--')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Val Acc - Train Acc')
axes[1, 1].set_title('Generalization Gap')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CFG.output_dir}/plots/training_history.png", dpi=150)
plt.show()
print("✓ Training history saved")

In [ ]:
# Confusion Matrix
cm = np.array(metrics['confusion_matrix'])
class_names = sorted(df_train['class_name'].unique())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
im1 = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].figure.colorbar(im1, ax=axes[0])
axes[0].set_xticks(range(len(class_names)))
axes[0].set_yticks(range(len(class_names)))
axes[0].set_xticklabels(class_names, rotation=45, ha='right')
axes[0].set_yticklabels(class_names)
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].set_title('Confusion Matrix (Counts)')

for i in range(len(class_names)):
    for j in range(len(class_names)):
        axes[0].text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im2 = axes[1].imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
axes[1].figure.colorbar(im2, ax=axes[1])
axes[1].set_xticks(range(len(class_names)))
axes[1].set_yticks(range(len(class_names)))
axes[1].set_xticklabels(class_names, rotation=45, ha='right')
axes[1].set_yticklabels(class_names)
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
axes[1].set_title('Confusion Matrix (Normalized)')

for i in range(len(class_names)):
    for j in range(len(class_names)):
        axes[1].text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center',
                    color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.savefig(f"{CFG.output_dir}/plots/confusion_matrix.png", dpi=150)
plt.show()
print("✓ Confusion matrix saved")

In [ ]:
# Per-class metrics
precision = metrics['precision_per_class']
recall = metrics['recall_per_class']
f1 = metrics['f1_per_class']

x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width, precision, width, label='Precision', alpha=0.8)
bars2 = ax.bar(x, recall, width, label='Recall', alpha=0.8)
bars3 = ax.bar(x + width, f1, width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Class')
ax.set_ylabel('Score')
ax.set_title('Per-Class Performance Metrics')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{CFG.output_dir}/plots/per_class_metrics.png", dpi=150)
plt.show()
print("✓ Per-class metrics saved")

## 8. XAI: Explainable AI Visualizations

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hooks = []
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        self.hooks.append(self.target_layer.register_forward_hook(forward_hook))
        self.hooks.append(self.target_layer.register_full_backward_hook(backward_hook))
    
    def generate(self, input_tensor, support_images, support_labels, target_class=None):
        self.model.eval()
        self.model.zero_grad()
        
        query_img = input_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits, _, _ = self.model(support_images, support_labels, query_img)
        
        if target_class is None:
            target_class = logits.argmax(dim=1).item()
        
        score = logits[0, target_class]
        score.backward()
        
        grads = self.gradients[0]
        acts = self.activations[0]
        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0)
        cam = F.relu(cam)
        cam = cam.cpu().numpy()
        
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = cv2.resize(cam, (input_tensor.shape[1], input_tensor.shape[2]))
        return cam
    
    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

def compute_saliency_map(model, input_tensor, support_images, support_labels, target_class=None):
    model.eval()
    query_img = input_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
    logits, _, _ = model(support_images, support_labels, query_img)
    if target_class is None:
        target_class = logits.argmax(dim=1).item()
    score = logits[0, target_class]
    score.backward()
    saliency = query_img.grad.data.abs().squeeze().cpu().numpy()
    saliency = np.max(saliency, axis=0)
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    return saliency

def visualize_explanation(image, mask, alpha=0.5):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = image.cpu().numpy().transpose(1, 2, 0)
    img_denorm = np.clip(img_np * std + mean, 0, 1)
    cmap = plt.get_cmap('jet')
    heatmap = cmap(mask)[:, :, :3]
    overlay = np.clip((1 - alpha) * img_denorm + alpha * heatmap, 0, 1)
    return img_denorm, heatmap, overlay

print("✓ XAI methods ready")

In [ ]:
# Generate XAI visualizations
print("\nGenerating XAI visualizations...")

test_dataset = FewShotDataset(df_test, transform=eval_transform)
# Use q_query_eval for XAI sampling
sampler = EpisodicSampler(df_test['label'].values, CFG.n_way, CFG.k_shot, 
                          CFG.q_query_eval, episodes=1, seed=CFG.seed)

support_idx, query_idx = next(iter(sampler))

support_data = [test_dataset[i] for i in support_idx]
support_images = torch.stack([x[0] for x in support_data]).to(DEVICE)
support_labels = torch.tensor([x[1] for x in support_data]).to(DEVICE)

unique_labels = torch.unique(support_labels)
label_map = {int(l): i for i, l in enumerate(unique_labels)}
support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], 
                                       dtype=torch.long, device=DEVICE)

gradcam = GradCAM(model, model.encoder.encoder[6])

n_samples = min(5, len(query_idx))
for i in range(n_samples):
    query_img, true_label = test_dataset[query_idx[i]]
    
    with torch.no_grad():
        logits, _, _ = model(support_images, support_labels_mapped, 
                            query_img.unsqueeze(0).to(DEVICE))
        pred_label = logits.argmax(dim=1).item()
    
    # Generate explanations
    gradcam_mask = gradcam.generate(query_img, support_images, support_labels_mapped, pred_label)
    saliency_mask = compute_saliency_map(model, query_img, support_images, support_labels_mapped, pred_label)
    
    # Visualize
    img_orig, heatmap_g, overlay_g = visualize_explanation(query_img, gradcam_mask)
    _, heatmap_s, overlay_s = visualize_explanation(query_img, saliency_mask)
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    axes[0, 0].imshow(img_orig)
    axes[0, 0].set_title(f'Original\nTrue: {true_label}, Pred: {pred_label}')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(heatmap_g)
    axes[0, 1].set_title('Grad-CAM Heatmap')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(overlay_g)
    axes[0, 2].set_title('Grad-CAM Overlay')
    axes[0, 2].axis('off')
    
    axes[1, 0].imshow(img_orig)
    axes[1, 0].set_title('Original')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(heatmap_s)
    axes[1, 1].set_title('Saliency Heatmap')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(overlay_s)
    axes[1, 2].set_title('Saliency Overlay')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{CFG.output_dir}/xai/sample_{i}_gradcam_saliency.png", dpi=150)
    plt.show()

gradcam.remove_hooks()
print(f"\n✓ XAI visualizations saved to {CFG.output_dir}/xai/")

## 9. Summary

All outputs have been saved to `/kaggle/working/`:

- **Model**: `checkpoints/best_model.pth`
- **Metrics**: `test_metrics.json`
- **Splits**: `splits/train.csv`, `val.csv`, `test.csv`
- **Plots**: `plots/training_history.png`, `confusion_matrix.png`, `per_class_metrics.png`
- **XAI**: `xai/sample_*.png`

## Mathematical Summary

| Component | Formula |
|-----------|---------|
| Prototype | $p_c = \frac{1}{|S_c|} \sum_{x_i \in S_c} f_\phi(x_i)$ |
| Distance | $d(z, p_c) = \|f_\phi(z) - p_c\|_2^2$ |
| Logits | $\ell_c = -d(z, p_c)$ |
| Loss | $\mathcal{L} = -\frac{1}{|Q|} \sum_{(z_j, y_j) \in Q} \log \frac{\exp(\ell_{y_j})}{\sum_c \exp(\ell_c)}$ |
| Accuracy | $\text{ACC} = \frac{1}{N} \sum_{i=1}^N \mathbf{1}(\hat{y}_i = y_i)$ |
| F1-Score | $\text{F1} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ |